# Baseline Comparison

Runs all baseline models on the same evaluation sets and produces a unified comparison table.

| Model | Type | Notes |
|---|---|---|
| Presidio | Rule-based | Microsoft open-source PHI anonymizer |
| GLiNER | Zero-shot NER | `urchade/gliner_mediumv2.1` |
| RoBERTa-i2b2 | Fine-tuned NER | `obi/deid_roberta_i2b2` |
| scispaCy | Medical NER | `en_core_sci_lg` (not de-id specific) |
| EdgeDeID-SFT | Ours | Qwen3-0.6B SFT |
| EdgeDeID-GRPO | Ours | Qwen3-0.6B SFT + GRPO |

## 0 — Setup

In [11]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
from tqdm import tqdm

from pipeline.data.preprocessing import preprocess_text
from pipeline.data.xml_utils import convert_to_xml
from pipeline.eval import compute_metrics, print_report, to_dataframe

# ── dataset selection ────────────────────────────────────────────────────────
# "synth" = held-out synthetic test set (LLM Eval.xlsx)
# "ext"   = external English benchmark
# "test"  = test.csv from HF dataset (data/test.csv)
DATASET = "test"   # change to "synth" or "ext" as needed

if DATASET == "synth":
    raw_df = pd.read_excel("../synth_eval/LLM Eval.xlsx")
    raw_df.rename(columns={"Revision": "default_response_1"}, inplace=True)
    raw_df = preprocess_text(raw_df, input_col="default_response_0", output_col="default_response_1")
    raw_df["GT-XML"] = raw_df["default_response_1"].apply(convert_to_xml)
    NOTE_COL = "default_response_0"
elif DATASET == "ext":
    raw_df = pd.read_csv("../ext_eval/en_data.csv")
    raw_df = preprocess_text(raw_df, input_col="Report", output_col="DeID")
    raw_df["GT-XML"] = raw_df["DeID"].apply(convert_to_xml)
    NOTE_COL = "Report"
elif DATASET == "test":
    raw_df = pd.read_csv("../../data/test.csv")
    print(raw_df.shape)
    raw_df = preprocess_text(raw_df, input_col="original", output_col="anonymized")
    raw_df["GT-XML"] = raw_df["anonymized"].apply(convert_to_xml)
    NOTE_COL = "original"
    print(raw_df.shape)
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Choose 'synth', 'ext', or 'test'.")

for tag in ["<anonymized_report>", "</anonymized_report>", "<anonymized_report/>",
            "<raw_report>", "</raw_report>", "<raw_report/>"]:
    raw_df[NOTE_COL] = raw_df[NOTE_COL].str.replace(tag, "", regex=False).str.strip()

notes     = raw_df[NOTE_COL].tolist()
gt_xmls   = raw_df["GT-XML"].tolist()
print(f"{len(notes)} evaluation samples | dataset={DATASET}")

(404, 3)
Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
  Entity       Detected  Reformatted
  ----------  ---------  -----------
  name              700            -
  age               332            -
  date             1061          466
  place               0            -
  address           214            -
  phone             230            9
  email             215            -
  id                232            -
  url               293            -
  device            228            -

Text processing pipeline completed!
(377, 4)
377 evaluation samples | dataset=test


In [2]:
raw_df.head()

,anonymized,original,examples,GT-XML
0,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","(2024), Liam Novak (URL: https://myhealthonlin...",NONE,<phi_entities>\n<names/>\n<ages/>\n<dates/>\n<...
1,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","On 2023, Liam Novak, a 68y male patient with a...",NONE,<phi_entities>\n<names/>\n<ages>\n<age>68</age...
2,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Dear Dr. Larsson, I hope this email finds you ...",NONE,<phi_entities>\n<names/>\n<ages>\n<age>46</age...
3,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Mr. Pt, email: d.kim1980@email.com, visited Me...",NONE,<phi_entities>\n<names/>\n<ages/>\n<dates>\n<d...
4,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","Mr. David Okafor, a 58year-old man residing at...",NONE,<phi_entities>\n<names/>\n<ages>\n<age>58</age...


In [ ]:
from pipeline.data.preprocessing import preprocess_text, inspect_changes
raw_df = pd.read_csv("../../data/test.csv")
before_col = raw_df['anonymized'].copy()          # snapshot before
df = preprocess_text(raw_df, input_col='original', output_col='anonymized')

# All changed rows (default 5 samples)
inspect_changes(before_col, df['anonymized'])

# Filter to a specific entity type
inspect_changes(before_col, df['anonymized'], entity='date', n=3)
inspect_changes(before_col, df['anonymized'], entity='phone')

# More/less context lines around each change
inspect_changes(before_col, df['anonymized'], entity='name', context=1)


In [10]:
raw_df = pd.read_csv("../../data/test.csv")
raw_df

,anonymized,original,examples
0,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","(2024), Liam Novak (URL: https://myhealthonlin...",NONE
1,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","On 2023, Liam Novak, a 68y male patient with a...",NONE
2,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Dear Dr. Larsson, I hope this email finds you ...",NONE
3,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Mr. Pt, email: d.kim1980@email.com, visited Me...",NONE
4,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","Mr. David Okafor, a 58year-old man residing at...",NONE
...,...,...,...
399,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","08-30-18, Pt, 68y man, presented to Cleveland ...",NONE
400,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","David Kim, a 28 years old male, visited the cl...",NONE
401,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","Mr. David Okafor, 28 years old, presented with...",NONE
402,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","David cKim, a 28 yo male, visited Mercy Hospit...",NONE


## 1 — Load Baseline Models

Comment out any baselines whose dependencies are not installed.

In [15]:
from baselines import PresidioBaseline, GLiNERBaseline, RobertaDeidBaseline, ScispaCyBaseline

models = [
    PresidioBaseline(score_threshold=0.5),
    GLiNERBaseline(model_name="urchade/gliner_mediumv2.1", threshold=0.5),
    RobertaDeidBaseline(model_name="obi/deid_roberta_i2b2"),
    #ScispaCyBaseline(model_name="en_core_sci_lg"),
]
print("Loaded:", [m.name for m in models])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Device set to use mps:0


Loaded: ['Presidio', 'GLiNER', 'RoBERTa-i2b2']


In [14]:
import os

# Set proxy for standard HTTP/HTTPS traffic
os.environ["http_proxy"] = "http://127.0.0.1:10809"
os.environ["https_proxy"] = "http://127.0.0.1:10809"

# Set fallback proxy for all other traffic (requires httpx[socks])
os.environ["all_proxy"] = "socks5://127.0.0.1:10808"

print("Proxy environment variables set.")

Proxy environment variables set.


## 2 — Run Baselines

In [16]:
all_results = {}   # model_name -> EvalResult

for model in models:
    print(f"\nRunning {model.name} ...")
    predictions = [
        model.predict_xml(note)
        for note in tqdm(notes, desc=model.name)
    ]
    result = compute_metrics(predictions, gt_xmls, source_notes=notes)
    all_results[model.name] = result
    print_report(result, title=model.name)


Running Presidio ...


Presidio: 100%|██████████| 377/377 [00:11<00:00, 31.45it/s]


 Presidio
  Samples evaluated : 377
  XML parse rate    : 1.0000

  Micro Precision   : 0.1662
  Micro Recall      : 0.3109
  Micro F1          : 0.2166
  Macro F1          : 0.2552

  Hallucination rate: 0.8338  (1 − Precision)
  Token reduction   : 0.1421

Tag                 P        R       F1       GT     Pred
------------------------------------------------------------
address        0.0000   0.0000   0.0000      214        0
age            0.0000   0.0000   0.0000      332        0
date           0.0462   0.1408   0.0696      412     1255
device         0.0000   0.0000   0.0000      228        0
email          0.9953   0.9953   0.9953      215      215
id             0.2222   0.0345   0.0597      232       36
name           0.0000   0.0000   0.0000        0     1651
phone          0.9597   0.5385   0.6899      221      124
place          0.1075   0.2188   0.1441      256      521
url            0.4220   0.9966   0.5929      293      692

Running GLiNER ...


GLiNER:   2%|▏         | 8/377 [00:01<00:55,  6.60it/s]/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 415 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
GLiNER:  12%|█▏        | 44/377 [00:06<00:58,  5.65it/s]/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 439 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
GLiNER:  15%|█▌        | 57/377 [00:09<00:47,  6.74it/s]/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1417 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b i

 GLiNER
  Samples evaluated : 377
  XML parse rate    : 1.0000

  Micro Precision   : 0.4504
  Micro Recall      : 0.6030
  Micro F1          : 0.5157
  Macro F1          : 0.5500

  Hallucination rate: 0.5496  (1 − Precision)
  Token reduction   : 0.1612

Tag                 P        R       F1       GT     Pred
------------------------------------------------------------
address        0.5652   0.9720   0.7148      214      368
age            0.9274   0.6536   0.7668      332      234
date           0.0293   0.0316   0.0304      412      443
device         0.6462   0.9211   0.7595      228      325
email          0.8318   0.4140   0.5528      215      107
id             0.6645   0.8966   0.7633      232      313
name           0.0000   0.0000   0.0000        0      696
phone          0.8171   0.9095   0.8608      221      246
place          0.5782   0.9531   0.7198      256      422
url            0.9365   0.2014   0.3315      293       63

Running RoBERTa-i2b2 ...


RoBERTa-i2b2: 100%|██████████| 377/377 [00:30<00:00, 12.16it/s]

 RoBERTa-i2b2
  Samples evaluated : 377
  XML parse rate    : 1.0000

  Micro Precision   : 0.3928
  Micro Recall      : 0.2722
  Micro F1          : 0.3215
  Macro F1          : 0.1508

  Hallucination rate: 0.6072  (1 − Precision)
  Token reduction   : 0.0361

Tag                 P        R       F1       GT     Pred
------------------------------------------------------------
address        0.0000   0.0000   0.0000      214        0
age            0.9736   1.0000   0.9866      332      341
date           0.2432   0.7816   0.3710      412     1324
device         0.0000   0.0000   0.0000      228        0
email          0.0000   0.0000   0.0000      215        0
id             0.0000   0.0000   0.0000      232        0
phone          0.0000   0.0000   0.0000      221        0
place          0.0000   0.0000   0.0000      256        0
url            0.0000   0.0000   0.0000      293        0


## 3 — Add Our Models

Load pre-computed predictions from the synth_eval / ext_eval notebooks
(run those notebooks first and save `*_predictions.csv`).

In [ ]:
import os

prefix = "synth_eval" if DATASET == "synth" else "ext_eval"

# SFT model predictions
sft_pred_path = f"../{prefix}/{prefix.replace('_eval','')}_eval_predictions.csv"
if os.path.exists(sft_pred_path):
    sft_df = pd.read_csv(sft_pred_path)
    sft_result = compute_metrics(
        sft_df["LLM-XML"].tolist(), gt_xmls, source_notes=notes
    )
    all_results["EdgeDeID-SFT"] = sft_result
    print_report(sft_result, title="EdgeDeID-SFT")
else:
    print(f"SFT predictions not found at {sft_pred_path} — run synth_eval.ipynb first.")

# GRPO model predictions — update path after GRPO training
# grpo_pred_path = f"../{prefix}/grpo_predictions.csv"
# if os.path.exists(grpo_pred_path):
#     grpo_df = pd.read_csv(grpo_pred_path)
#     grpo_result = compute_metrics(grpo_df["LLM-XML"].tolist(), gt_xmls, source_notes=notes)
#     all_results["EdgeDeID-GRPO"] = grpo_result
#     print_report(grpo_result, title="EdgeDeID-GRPO")

SFT predictions not found at ../ext_eval/ext_eval_predictions.csv — run synth_eval.ipynb first.


## 4 — Comparison Table

In [17]:
rows = []
for model_name, r in all_results.items():
    rows.append({
        "Model":              model_name,
        "Micro-P":            round(r.micro_precision, 4),
        "Micro-R":            round(r.micro_recall, 4),
        "Micro-F1":           round(r.micro_f1, 4),
        "Macro-F1":           round(r.macro_f1, 4),
        "XML Parse Rate":     round(r.xml_parse_rate, 4),
        "Hallucination Rate": round(r.hallucination_rate, 4),
        "Token Reduction":    round(r.token_reduction_rate, 4) if r.token_reduction_rate == r.token_reduction_rate else None,
    })

comparison_df = pd.DataFrame(rows).set_index("Model")
comparison_df

,Micro-P,Micro-R,Micro-F1,Macro-F1,XML Parse Rate,Hallucination Rate,Token Reduction
Model,,,,,,,
Presidio,0.1662,0.3109,0.2166,0.2552,1.0,0.8338,0.1421
GLiNER,0.4504,0.6030,0.5157,0.5500,1.0,0.5496,0.1612
RoBERTa-i2b2,0.3928,0.2722,0.3215,0.1508,1.0,0.6072,0.0361


In [20]:
import os
os.makedirs("results", exist_ok=True)

out_path = f"results/baseline_comparison_{DATASET}.csv"
comparison_df.to_csv(out_path)
print(f"Saved {out_path}")

Saved results/baseline_comparison_test.csv


## 5 — Per-Entity Breakdown

In [21]:
all_entity_tags = sorted(
    set(tag for r in all_results.values() for tag in r.per_entity)
)
per_entity_rows = []
for entity in all_entity_tags:
    row = {"Entity": entity}
    for model_name, r in all_results.items():
        m = r.per_entity.get(entity)
        row[model_name] = round(m.f1, 4) if m else None
    per_entity_rows.append(row)

per_entity_df = pd.DataFrame(per_entity_rows).set_index("Entity")

out_path = f"results/baseline_per_entity_{DATASET}.csv"
per_entity_df.to_csv(out_path)
print(f"Saved {out_path}")
per_entity_df

Saved results/baseline_per_entity_test.csv


,Presidio,GLiNER,RoBERTa-i2b2
Entity,,,
address,0.0000,0.7148,0.0000
age,0.0000,0.7668,0.9866
date,0.0696,0.0304,0.3710
device,0.0000,0.7595,0.0000
email,0.9953,0.5528,0.0000
id,0.0597,0.7633,0.0000
name,0.0000,0.0000,NaN
phone,0.6899,0.8608,0.0000
place,0.1441,0.7198,0.0000
